# Using Tools with Claude

In [2]:
from dotenv import load_dotenv
import anthropic
from anthropic import Anthropic
from rich.console import Console

print(f"Using Anthropic API: {anthropic.__version__}")

load_dotenv(override=True)

client = Anthropic()
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 1024

console = Console(force_jupyter=False)

Using Anthropic API: 1.1.0


## Modified Helper functions

We'll have to modify our `add_user_message()`, `add_assistant_message()` and `chat()` functions to handle multiple tool calls.

These have been added to the `my_chat_utils_with_tools.py` file.

In [3]:
from my_chat_utils_with_tools import (
    add_assistant_message,
    add_user_message,
    chat,
    text_from_message,
)

### Define the tool functions

Let's define a tool function to get current date & time in a given format.

We'll define the following functions in the `my_tool_functions.py` file
* `get_current_datetime`
* `add_duration_to_datetime`
* `set_reminder`

> 📌 **NOTE**: When you want to add a new function, go to the `my_tool_functions.py` file and add function definition + append function name to imports below.

In [4]:
from my_tool_functions import (
    get_current_datetime,
    add_duration_to_datetime,
    set_reminder,
)

## Defining tool schemas

We will also need to create a JSON schema describing the tool call function & params. This can be generated using Claude AI. Following schema was generated by Claude AI, which we assign to a varible, named with the same name as the tool function and ending with `_schema`.


An easy way to create the schema is to ask Claude to generate it. Head over to [claude.ai](https://claude.ai) and type in the following prompt and paste the tool function below the prompt:

`"Write a valid JSON schema spec for the purposes of tool calling for this function. Follow the best practices listed in the attached documentation available at https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview.md"`

📌 **NOTE**: as of Aug 2026, the URL for Anthropic's tool documentation is - `https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview.md` - this could change in future. Paste in the correct URL.

These schemas are saved in `my_tool_schemas.py` code file.

> 📌 **NOTE**: If you add a new tool function, create it's corresponding schema & add it to the `my_tool_schemas.py` file.

In [5]:
from my_tool_schemas import (
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
    batch_tool_schema,
)

Now let's test the tool functions.

In [6]:
# some test calls
print(f"Default: {get_current_datetime()}")
print(f"Custom: {get_current_datetime('%d/%m/%Y %I:%M %p')}")

date_format = "%Y-%m-%d"  # "%Y-%m-%d %H:%M:%S"
datetime_str = f"{get_current_datetime(date_format)}"
print(datetime_str)
print(add_duration_to_datetime(datetime_str, 5, "days", date_format))
print(add_duration_to_datetime(datetime_str, 1, "months", date_format))

Default: 2026-09-23 10:19:49
Custom: 23/09/2026 10:19 AM
2026-09-23
Monday, September 28, 2026 12:00:00 AM
Friday, October 23, 2026 12:00:00 AM


Utility functions to run a conversation with tools.

> 📌 **NOTE**: to call any new tool you add, add the function call to the `run_tool` function below. Also add the tool schema to the `tool_schemas` list.

In [11]:
import json

tool_schemas = [
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
]


def run_tool(tool_name, tool_input):
    """runs a specific tools with the tool params"""
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)


def run_tools(message):
    """finds out tools from message block and runs the tools"""
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks


def run_conversation(messages):
    while True:
        response = chat(
            client,
            MODEL,
            messages,
            tools=tool_schemas,
        )

        add_assistant_message(messages, response)
        # added this print so we can follow along
        console.print(
            f"[blue]Got response from chat:[/blue] {response}\n[yellow]-----[/yellow]"
        )
        text_response = text_from_message(response)
        # added this print so we can follow along
        console.print(f"[green]text_from_message:[/green] {text_response}\n")
        console.print(
            f"[red]Stop reason:[/red] {response.stop_reason}\n[yellow]-----[/yellow]"
        )

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

Now let's see how Claude calls a single tool

In [12]:
messages = []

add_user_message(
    messages,
    "What is the exact time formatted as HH:MM:SS?",
)
run_conversation(messages)

Got response from chat: Message(id='msg_011CfLDAQTEk4ZC184ZXHFt9', 
container=None, content=[ToolUseBlock(id='toolu_019mqfBFygAkmLhmDpMMyxLH', 
caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, 
name='get_current_datetime', type='tool_use', toolset_name=None)], 
model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, 
stop_reason='tool_use', stop_sequence=None, type='message', 
usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, 
ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, 
cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1544, 
output_tokens=63, output_tokens_details=None, server_tool_use=None, 
service_tier='standard'))
-----
text_from_message: 

Stop reason: tool_use
-----
Got response from chat: Message(id='msg_011CfLDAVPe74v3PmhTV3bAo', 
container=None, content=[TextBlock(citations=None, text='The exact time is 
**10:26:13** (HH:MM:SS format).', type='text')], 
model='claude-haiku-4-5-202

[{'role': 'user', 'content': 'What is the exact time formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_019mqfBFygAkmLhmDpMMyxLH', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_019mqfBFygAkmLhmDpMMyxLH',
    'content': '"10:26:13"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The exact time is **10:26:13** (HH:MM:SS format).', type='text')]}]

And now let's try our set appointment message

In [13]:
messages = []
add_user_message(
    messages,
    "Set reminder for my doctors appointment. It's 177 days after Jan 1st, 2026.",
)
run_conversation(messages)

Got response from chat: Message(id='msg_011CfLDEMVk4G8s8g8miFa7B', 
container=None, content=[TextBlock(citations=None, text='I need to calculate 
the date that is 177 days after January 1st, 2026, and then set a reminder for 
that date.', type='text'), ToolUseBlock(id='toolu_01J8xieUumCcbKQGyMSoipDS', 
caller=DirectCaller(type='direct'), input={'datetime_str': '2026-01-01', 
'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', 
type='tool_use', toolset_name=None)], model='claude-haiku-4-5-20251001', 
role='assistant', stop_details=None, stop_reason='tool_use', 
stop_sequence=None, type='message', 
usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, 
ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, 
cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1553, 
output_tokens=130, output_tokens_details=None, server_tool_use=None, 
service_tier='standard'))
-----
text_from_message: I need to calculate the date that is 177 days aft

[{'role': 'user',
  'content': "Set reminder for my doctors appointment. It's 177 days after Jan 1st, 2026."},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='I need to calculate the date that is 177 days after January 1st, 2026, and then set a reminder for that date.', type='text'),
   ToolUseBlock(id='toolu_01J8xieUumCcbKQGyMSoipDS', caller=DirectCaller(type='direct'), input={'datetime_str': '2026-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01J8xieUumCcbKQGyMSoipDS',
    'content': '"Saturday, June 27, 2026 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on June 27, 2026:", type='text'),
   ToolUseBlock(id='toolu_017SvVV2U1EsHmnY8neLVjoE', caller=DirectCaller(type='direct'), input={'content': "Doctor